# Data Engineering Analytics

This notebook reads the SQLite data generated by `pipeline_etl.py` and answers the required analytical questions.

In [6]:
import sqlite3
import pandas as pd

DB_PATH = "data/sunrise_sunset.db"
conn = sqlite3.connect(DB_PATH)

check_df = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM sunrise_sunset", conn)
check_df

,total_rows
0,2254


## 1) Longest and Shortest Day (Per Calendar Year)

In [ ]:
base_df = pd.read_sql_query(
    "SELECT date, day_length_seconds FROM sunrise_sunset ORDER BY date",
    conn,
)

base_df["year"] = pd.to_datetime(base_df["date"]).dt.year

longest_df = base_df.loc[
    base_df.groupby("year")["day_length_seconds"].idxmax(),
    ["year", "date"],
]
longest_df = longest_df.rename(columns={"date": "longest_day_date"})

shortest_df = base_df.loc[
    base_df.groupby("year")["day_length_seconds"].idxmin(),
    ["year", "date"],
]
shortest_df = shortest_df.rename(columns={"date": "shortest_day_date"})

result_long_short = (
    longest_df.merge(shortest_df, on="year")
    .sort_values("year")
    .reset_index(drop=True)
)

result_long_short

,year,longest_day_date,longest_day_seconds,shortest_day_date,shortest_day_seconds
0,2020,2020-06-20,47952,2020-12-21,39679
1,2021,2021-06-20,47952,2021-12-21,39679
2,2022,2022-06-20,47952,2022-12-21,39679
3,2023,2023-06-20,47952,2023-12-22,39679
4,2024,2024-06-20,47952,2024-12-21,39679
5,2025,2025-06-20,47952,2025-12-21,39679
6,2026,2026-03-03,42632,2026-01-01,39765


## 2) Latest Sunrise Time (Per Month)

In [ ]:
query_latest_sunrise = """
SELECT
  strftime('%Y-%m', date) AS year_month,
  MAX(time(sunrise_ist)) AS latest_sunrise_time_ist
FROM sunrise_sunset
GROUP BY 1
ORDER BY 1;
"""

pd.read_sql_query(query_latest_sunrise, conn)

,year_month,latest_sunrise_time_ist
0,2026-03,01:26:37


## 3) Earliest Sunset Time (Per Month)

In [ ]:
query_earliest_sunset = """
SELECT
  strftime('%Y-%m', date) AS year_month,
  MIN(time(sunset_ist)) AS earliest_sunset_time_ist
FROM sunrise_sunset
GROUP BY 1
ORDER BY 1;
"""

pd.read_sql_query(query_earliest_sunset, conn)

,year_month,earliest_sunset_time_ist
0,2026-03,13:15:02


In [ ]:
conn.close()